In [2]:
import cv2
import numpy as np
import math
import os
import pandas as pd

In [9]:
step = 65.5
dt = step/1000 # dt in second fixed for all frames

# Function to get coordinates of the red point in an image
def get_red_point_coordinates(image_path):
    # Read the image
    image = cv2.imread(image_path)

    # Convert the image from BGR to HSV
    hsv_image = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Define the range of red color in HSV
    lower_red = np.array([0, 100, 100])
    upper_red = np.array([10, 255, 255])

    # Create a mask using the inRange function
    mask = cv2.inRange(hsv_image, lower_red, upper_red)

    # Find contours in the mask
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Check if any contours are found
    if contours:
        # Get the largest contour (assuming the red point is the biggest red region)
        largest_contour = max(contours, key=cv2.contourArea)

        # Get the centroid of the largest contour
        M = cv2.moments(largest_contour)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])

            # Return the coordinates
            return cx, cy
        else:
            print(f"Unable to determine coordinates for {image_path}.")
    else:
        print(f"No red point found in {image_path}.")
        return None

# Path to the folder containing the images
folder_path = 'frameImage'

# Create a list to store the image paths
image_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.png')]

# Create a DataFrame to store the velocity
df_velocity = pd.DataFrame(columns=['velocityX', 'velocityY', 'velocity'])

# Create a new image with the same size as the first image
first_image = cv2.imread(image_paths[0])
combined_image = np.zeros_like(first_image)

# Iterate through the images and calculate distances
for i in range(len(image_paths)-1):
    # Get coordinates of the red points for the current and next images
    red_point_current = get_red_point_coordinates(image_paths[i])
    red_point_next = get_red_point_coordinates(image_paths[i+1])

    if red_point_current is not None and red_point_next is not None:
        # Calculate distances
        distanceX = red_point_next[0] - red_point_current[0]
        distanceY = red_point_next[1] - red_point_current[1]
        distance = np.sqrt(distanceX**2 + distanceY**2)
        vx = abs(distanceX/dt)
        vy = abs(distanceY/dt)
        v = math.sqrt(vx**2 + vy**2)


        # Append velocity to the DataFrame
        df_velocity = df_velocity.append({'velocityX': vx, 'velocityY': vy, 'velocity': v}, ignore_index=True)

        # Draw a line on the combined image
        cv2.line(combined_image, red_point_current, red_point_next, (0, 255, 0), 2)  # (0, 255, 0) is green color

# Save the combined image
cv2.imwrite('frameImage/combined_image_with_lines.jpg', combined_image)

# Save the DataFrame to a CSV file
df_velocity.to_csv('real_velocity.csv', index=False)

# Display the DataFrame
df_velocity


C:\Users\DELL\AppData\Local\Temp\ipykernel_4604\3416981518.py:71: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_velocity = df_velocity.append({'velocityX': vx, 'velocityY': vy, 'velocity': v}, ignore_index=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_4604\3416981518.py:71: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_velocity = df_velocity.append({'velocityX': vx, 'velocityY': vy, 'velocity': v}, ignore_index=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_4604\3416981518.py:71: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_velocity = df_velocity.append({'velocityX': vx, 'velocityY': vy, 'velocity': v}, ignore_index=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_4604\3416981518.py:71: FutureWarning: The fra

,velocityX,velocityY,velocity
0,625.954198,76.335878,630.591647
1,106.870229,5068.702290,5069.828809
2,15.267176,30.534351,34.138442
3,0.000000,76.335878,76.335878
4,30.534351,91.603053,96.558096
...,...,...,...
177,15.267176,91.603053,92.866604
178,0.000000,106.870229,106.870229
179,30.534351,61.068702,68.276885
180,0.000000,106.870229,106.870229
